# E-Commerce Sales & Customer Analytics

In [3]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

OUT = "outputs"
CHARTS = os.path.join(OUT, "charts")

os.makedirs(CHARTS, exist_ok=True)

## 1. Load DataLoad the master `combined_dataset_reduced.csv` and parse order dates.

In [7]:
df = pd.read_csv("combined_dataset_reduced.csv", low_memory=False)

df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

print(
    f"Loaded combined_dataset_reduced: "
    f"{df.shape[0]:,} rows x {df.shape[1]} cols"
)

Loaded combined_dataset_reduced: 198,785 rows x 58 cols


## 2. Descriptive StatisticsCompute mean, median, mode, standard deviation, variance and skew for every key numeric column, and save the results to `summary_statistics.csv`.

In [9]:
numeric_cols = [
    "quantity",
    "unit_price",
    "discount_percentage",
    "discount_amount",
    "gross_sales",
    "tax_amount",
    "shipping_cost",
    "net_sales",
    "product_cost",
    "profit",
    "profit_margin_percentage",
    "customer_age",
    "customer_rating",
    "product_rating",
    "delivery_days",
    "customer_lifetime_value",
    "customer_acquisition_cost",
    "loyalty_points_earned"
]

summary_rows = []

for col in numeric_cols:
    s = df[col].dropna()
    mode_val = s.mode()

    summary_rows.append({
        "column": col,
        "mean": round(s.mean(), 3),
        "median": round(s.median(), 3),
        "mode": round(mode_val.iloc[0], 3) if not mode_val.empty else np.nan,
        "std_dev": round(s.std(), 3),
        "variance": round(s.var(), 3),
        "min": round(s.min(), 3),
        "max": round(s.max(), 3),
        "skew": round(s.skew(), 3),
        "count": s.shape[0],
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    f"{OUT}/summary_statistics.csv",
    index=False
)

print("\n=== DESCRIPTIVE STATISTICS (mean / median / mode) ===")
print(summary_df.to_string(index=False))


=== DESCRIPTIVE STATISTICS (mean / median / mode) ===
                   column     mean   median     mode  std_dev     variance      min      max   skew  count
                 quantity    2.115    2.000     1.00    1.193        1.423     1.00     5.00  0.805 198785
               unit_price  245.143  164.970    59.25  251.053    63027.431     6.31  1482.95  1.989 198785
      discount_percentage    0.147    0.124     0.00    0.130        0.017     0.00     0.60  1.186 198785
          discount_amount   89.384   26.210     0.00  189.791    36020.518     0.00  3581.65  5.391 198785
              gross_sales  517.629  279.170   111.12  673.205   453204.512     6.31  7414.75  3.188 198785
               tax_amount   46.064   22.530     4.54   70.956     5034.769     0.26  1482.95  4.655 198785
            shipping_cost    8.414    5.870     0.00    6.788       46.072     0.00    36.46  1.189 198785
                net_sales  482.723  270.490    74.62  612.668   375361.802     5.35  8897

## 3. Correlation AnalysisPearson correlation matrix across key numeric variables, visualized as a heatmap, plus the variables most correlated with `profit`.

In [11]:
corr_cols = [
    "quantity",
    "unit_price",
    "discount_percentage",
    "gross_sales",
    "net_sales",
    "profit",
    "profit_margin_percentage",
    "customer_age",
    "customer_rating",
    "product_rating",
    "delivery_days",
    "customer_lifetime_value",
    "customer_acquisition_cost"
]

# Calculate Pearson correlation matrix
corr_matrix = df[corr_cols].corr(method="pearson")

# Save correlation matrix
corr_matrix.to_csv(
    f"{OUT}/correlation_matrix.csv"
)

# Create heatmap
plt.figure(figsize=(11, 9))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.title(
    "Correlation Matrix — Key Numeric Variables",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/01_correlation_heatmap.png"
)

plt.close()

# Top correlations with profit
profit_corr = (
    corr_matrix["profit"]
    .drop("profit")
    .sort_values(key=abs, ascending=False)
)

print("\n=== TOP CORRELATIONS WITH PROFIT ===")
print(profit_corr.to_string())


=== TOP CORRELATIONS WITH PROFIT ===
net_sales                    0.895881
gross_sales                  0.783212
unit_price                   0.659293
quantity                     0.333932
discount_percentage         -0.202610
profit_margin_percentage     0.123280
customer_lifetime_value      0.122254
product_rating              -0.014329
customer_rating              0.004109
delivery_days                0.001390
customer_acquisition_cost   -0.001151
customer_age                 0.000205


## 4. VisualizationsNine charts covering sales by category/brand/channel, monthly trend, profit-margin distribution, customer age by segment, discount vs. margin, and return-status breakdown.

In [13]:
# 4a. Net sales by product category
cat_sales = (
    df.groupby("product_category")["net_sales"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=cat_sales.values,
    y=cat_sales.index,
    hue=cat_sales.index,
    palette="viridis",
    legend=False
)

plt.title(
    "Net Sales by Product Category",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Net Sales ($)")
plt.tight_layout()

plt.savefig(
    f"{CHARTS}/02_sales_by_category.png"
)

plt.close()


# 4b. Monthly sales trend
monthly = (
    df.dropna(subset=["order_date"])
    .set_index("order_date")
    .resample("ME")["net_sales"]
    .sum()
)

plt.figure(figsize=(12, 5))

plt.plot(
    monthly.index,
    monthly.values,
    marker="o",
    color="darkblue",
    linewidth=1.5
)

plt.title(
    "Monthly Net Sales Trend",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel("Net Sales ($)")
plt.xlabel("Month")

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/03_monthly_sales_trend.png"
)

plt.close()


# 4c. Profit margin distribution
plt.figure(figsize=(9, 5))

sns.histplot(
    df["profit_margin_percentage"].dropna(),
    bins=60,
    kde=True,
    color="teal"
)

plt.axvline(
    df["profit_margin_percentage"].mean(),
    color="red",
    linestyle="--",
    label="Mean"
)

plt.axvline(
    df["profit_margin_percentage"].median(),
    color="orange",
    linestyle="--",
    label="Median"
)

plt.title(
    "Distribution of Profit Margin %",
    fontsize=14,
    fontweight="bold"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    f"{CHARTS}/04_profit_margin_distribution.png"
)

plt.close()


# 4d. Customer age distribution by segment
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="customer_segment",
    y="customer_age",
    hue="customer_segment",
    palette="Set2",
    legend=False
)

plt.title(
    "Customer Age by Segment",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/05_age_by_segment.png"
)

plt.close()


# 4e. Sales channel performance
channel = (
    df.groupby("sales_channel")
    .agg(
        orders=("order_id", "nunique"),
        net_sales=("net_sales", "sum"),
        avg_rating=("customer_rating", "mean")
    )
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=channel,
    x="sales_channel",
    y="net_sales",
    hue="sales_channel",
    ax=ax1,
    palette="mako",
    legend=False
)

ax1.set_title(
    "Net Sales by Sales Channel",
    fontsize=14,
    fontweight="bold"
)

ax1.set_ylabel("Net Sales ($)")

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/06_sales_by_channel.png"
)

plt.close()


# 4f. Discount vs Profit Margin
sample = df.sample(
    min(8000, len(df)),
    random_state=42
)

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=sample,
    x="discount_percentage",
    y="profit_margin_percentage",
    alpha=0.3,
    s=15,
    color="purple"
)

sns.regplot(
    data=sample,
    x="discount_percentage",
    y="profit_margin_percentage",
    scatter=False,
    color="red",
    line_kws={"linewidth": 2}
)

plt.title(
    "Discount % vs Profit Margin % (sampled)",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/07_discount_vs_margin.png"
)

plt.close()


# 4g. Top 10 brands by revenue
brand_sales = (
    df.groupby("brand")["net_sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=brand_sales.values,
    y=brand_sales.index,
    hue=brand_sales.index,
    palette="crest",
    legend=False
)

plt.title(
    "Top 10 Brands by Net Sales",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Net Sales ($)")

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/08_top_brands.png"
)

plt.close()


# 4h. Return status breakdown
plt.figure(figsize=(7, 7))

ret = df["return_status"].value_counts()

plt.pie(
    ret.values,
    labels=ret.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("pastel")
)

plt.title(
    "Return Status Breakdown (of returned orders)",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/09_return_status.png"
)

plt.close()


print(f"\nSaved 8 charts to {CHARTS}")


Saved 8 charts to outputs/charts


## 5. Machine Learning**Model A (Regression):** predict `profit` from order-level features using Linear Regression and Random Forest.**Model B (Classification):** predict `is_repeat_customer` from customer-level features using Random Forest.> **Caveat:** Model A's top features (`tax_amount`, `gross_sales`) are near-arithmetic derivatives of `profit` in this dataset, and Model B's `customer_order_count` directly encodes the repeat-customer label — both models exhibit **data leakage** and should be rebuilt without these fields for a genuine predictive signal (see the last cell for a leakage-free version).

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)

results_txt = []


# ============================================================
# MODEL A: REGRESSION — PREDICT PROFIT
# ============================================================

ml_df = df[
    [
        "quantity",
        "unit_price",
        "discount_percentage",
        "gross_sales",
        "tax_amount",
        "shipping_cost",
        "product_cost",
        "customer_age",
        "customer_rating",
        "product_rating",
        "profit_margin_percentage",
        "product_category",
        "sales_channel",
        "profit"
    ]
].dropna()

cat_features = [
    "product_category",
    "sales_channel"
]

le_dict = {}

for c in cat_features:
    le = LabelEncoder()
    ml_df[c + "_enc"] = le.fit_transform(ml_df[c])
    le_dict[c] = le


feature_cols = [
    "quantity",
    "unit_price",
    "discount_percentage",
    "gross_sales",
    "tax_amount",
    "shipping_cost",
    "product_cost",
    "customer_age",
    "customer_rating",
    "product_rating",
    "product_category_enc",
    "sales_channel_enc"
]

X = ml_df[feature_cols]
y = ml_df["profit"]


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Linear Regression
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)


# Random Forest Regression
rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)


# Evaluate regression models
results_txt.append(
    "=== MODEL A: Regression — Predicting `profit` ===\n"
)

for name, pred in [
    ("Linear Regression", lr_pred),
    ("Random Forest", rf_pred)
]:

    mae = mean_absolute_error(y_test, pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    r2 = r2_score(y_test, pred)

    results_txt.append(
        f"{name}: "
        f"MAE={mae:.2f}  "
        f"RMSE={rmse:.2f}  "
        f"R2={r2:.4f}"
    )


# Random Forest feature importance
importances = (
    pd.Series(
        rf.feature_importances_,
        index=feature_cols
    )
    .sort_values(ascending=False)
)

results_txt.append(
    "\nRandom Forest Feature Importances "
    "(predicting profit):"
)

results_txt.append(
    importances.to_string()
)


# Feature importance chart
plt.figure(figsize=(9, 6))

sns.barplot(
    x=importances.values,
    y=importances.index,
    hue=importances.index,
    palette="flare",
    legend=False
)

plt.title(
    "Feature Importance — Predicting Profit "
    "(Random Forest)",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    f"{CHARTS}/10_feature_importance_profit.png"
)

plt.close()


# ============================================================
# MODEL B: CLASSIFICATION — PREDICT REPEAT CUSTOMER
# ============================================================

ml_df2 = df[
    [
        "customer_age",
        "customer_order_count",
        "customer_lifetime_value",
        "customer_acquisition_cost",
        "loyalty_points_earned",
        "loyalty_points_redeemed",
        "customer_rating",
        "profit_margin_percentage",
        "customer_segment",
        "is_repeat_customer"
    ]
].dropna()


# Encode customer segment
le_seg = LabelEncoder()

ml_df2["customer_segment_enc"] = (
    le_seg.fit_transform(
        ml_df2["customer_segment"]
    )
)


feat_b = [
    "customer_age",
    "customer_order_count",
    "customer_lifetime_value",
    "customer_acquisition_cost",
    "loyalty_points_earned",
    "loyalty_points_redeemed",
    "customer_rating",
    "profit_margin_percentage",
    "customer_segment_enc"
]

Xb = ml_df2[feat_b]

yb = ml_df2["is_repeat_customer"].astype(int)


# Train/test split
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb,
    yb,
    test_size=0.2,
    random_state=42,
    stratify=yb
)


# Random Forest Classifier
clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

clf.fit(Xb_train, yb_train)

yb_pred = clf.predict(Xb_test)


# Classification results
results_txt.append(
    "\n\n=== MODEL B: Classification — "
    "Predicting `is_repeat_customer` ==="
)

results_txt.append(
    f"Accuracy: "
    f"{accuracy_score(yb_test, yb_pred):.4f}"
)

results_txt.append(
    classification_report(
        yb_test,
        yb_pred
    )
)


# Classification feature importance
importances_b = (
    pd.Series(
        clf.feature_importances_,
        index=feat_b
    )
    .sort_values(ascending=False)
)

results_txt.append(
    "Feature Importances "
    "(predicting repeat customer):"
)

results_txt.append(
    importances_b.to_string()
)


# ============================================================
# SAVE RESULTS
# ============================================================

with open(
    f"{OUT}/ml_model_results.txt",
    "w"
) as f:

    f.write(
        "\n".join(results_txt)
    )


print(
    "\n".join(results_txt)
)

print(
    f"\nAll outputs written to {OUT}"
)

=== MODEL A: Regression — Predicting `profit` ===

Linear Regression: MAE=49.08  RMSE=90.53  R2=0.8729
Random Forest: MAE=11.36  RMSE=28.36  R2=0.9875

Random Forest Feature Importances (predicting profit):
tax_amount              0.665498
gross_sales             0.148955
discount_percentage     0.135811
product_cost            0.044702
unit_price              0.000936
product_rating          0.000813
shipping_cost           0.000744
customer_age            0.000637
customer_rating         0.000608
quantity                0.000586
product_category_enc    0.000440
sales_channel_enc       0.000269


=== MODEL B: Classification — Predicting `is_repeat_customer` ===
Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        77
           1       1.00      1.00      1.00     32618

    accuracy                           1.00     32695
   macro avg       1.00      1.00      1.00     32695
weighted avg       1.00      1.00      1

## 6. Bonus: Leakage-Free Profit ModelA cleaner regression that excludes derived/leaky fields (`tax_amount`, `gross_sales`, `net_sales`) and only uses pre-sale, decision-time features.

In [18]:
from sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import mean_absolute_error, r2_scoreclean_feats = ["quantity", "unit_price", "discount_percentage", "product_cost",               "customer_age", "product_rating", "product_category_enc", "sales_channel_enc"]Xc = ml_df[clean_feats]yc = ml_df["profit"]Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42)rf_clean = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)rf_clean.fit(Xc_train, yc_train)pred_clean = rf_clean.predict(Xc_test)print(f"Leakage-free Random Forest: MAE={mean_absolute_error(yc_test, pred_clean):.2f}  R2={r2_score(yc_test, pred_clean):.4f}")pd.Series(rf_clean.feature_importances_, index=clean_feats).sort_values(ascending=False)

SyntaxError: invalid syntax (1256481051.py, line 1)